# HC3 Data Collection and preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# import the json file downloaded manually
import pandas as pd
df =  pd.read_json("/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/all.json", lines=True)
df["id"] = range(0, len(df))

In [ ]:
df = df[["id","question","human_answers", "chatgpt_answers", "source"]]
df

,id,question,human_answers,chatgpt_answers,source
0,0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,reddit_eli5
1,1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,reddit_eli5
2,2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,reddit_eli5
3,3,Why has nobody assassinated Kim Jong - un He i...,[You ca n't just go around assassinating the l...,[It is generally not acceptable or ethical to ...,reddit_eli5
4,4,How was airplane technology able to advance so...,[Wanting to kill the shit out of Germans drive...,[After the Wright Brothers made the first powe...,reddit_eli5
...,...,...,...,...,...
24317,24317,Is rise in pressure from 116/66 to 140/80 norm...,[Hello!Welcome and thank you for asking on HCM...,[It's not uncommon for blood pressure to fluct...,medicine
24318,24318,What could cause a painless lump in the right ...,"[Hi, * As per my surgical experience, the issu...",[There are several possible causes of a painle...,medicine
24319,24319,Can Acutret be given to a child for treatment ...,[Although it is difficult to comment whether A...,[It is not appropriate for me to recommend a s...,medicine
24320,24320,Are BP of 119/65 and pulse of 35 causes for co...,[Welcome and thank you for asking on HCM! I ha...,[It is not uncommon for people with rheumatoid...,medicine


In [ ]:
def separate_dataset(df_train):#separate dataset at the fraction of 1 : 1 : 8 in each source into test, validation, and train data.
  #computing the fraction on each source to make equally distributed data.
  # making test dataset
  source_list = ['open_qa', 'wiki_csai', 'finance', 'medicine']
  df_source = df[df.source == 'reddit_eli5']
  df_test = df_source.sample(frac = 0.1, random_state= 1)
  train_drop_ids = list(df_test['id'].astype("int64").values)
  for source in source_list:
    df_source = df[df.source == source]
    df_source_test = df_source.sample(frac = 0.1, random_state= 1)
    df_test = pd.concat([df_test,df_source_test])
    train_drop_ids.extend(list(df_source_test['id'].astype("int64").values))
 #remove the test data id from full data
 # the data contains train and validation data.
  df_train = df.drop(train_drop_ids)

  # making val dataset the same codes as test data making except for the fraction of split
  df_source = df_train[df_train.source == 'reddit_eli5']
  df_val = df_source.sample(frac = 1/9, random_state= 1)
  drop_ids = list(df_val['id'].astype("int64").values)
  for source in source_list:
    df_source = df_train[df_train.source == source]
    df_source_val = df_source.sample(frac = 1/9, random_state= 1)
    df_val = pd.concat([df_val,df_source_val])
    drop_ids.extend(list(df_source_val['id'].astype("int64").values))

  #remove the validataion data id from train and validation data.
  df_train = df_train.drop(drop_ids)

  return df_test, df_val, df_train, train_drop_ids


In [ ]:
# test data : validation data : training data = 8 : 1 : 1
df_test, df_val, df_train, train_drop_list = separate_dataset(df) # separate with the source

In [ ]:
#input: id, question, human_answers, chatgpt_answers
# output: text, label(human = 0, gpt= 1)
def to_bert_input(df):
  df_human = df[['id', 'question', 'human_answers','source']]
  df_human = df_human.explode('human_answers')
  df_human = df_human.rename(columns = {"human_answers":"text"})
  df_human = df_human.assign(label = 0)
  df_gpt = df[['id', 'question', 'chatgpt_answers', 'source']]
  df_gpt = df_gpt.explode('chatgpt_answers')
  df_gpt = df_gpt.rename(columns = {"chatgpt_answers":"text"})
  df_gpt = df_gpt.assign(label = 1)
  return pd.concat([df_human[['text','label', 'source']],df_gpt[['text','label', 'source']]])

In [ ]:
# applying the function above into train, validation, test data.
test_data = to_bert_input(df_test)
val_data = to_bert_input(df_val)
train_data = to_bert_input(df_train)

In [ ]:
# checking the category distribution for the analysis
def check_category_numbers(test, val, train):
  answers = []
  sources_list = ['reddit_eli5', 'open_qa', 'wiki_csai','medicine','finance']
  # in each source, sum up the number of texts from test, val, and train sets
  for source in sources_list:
    a = test[(test["source"] == source)].dropna()
    b = train[(train["source"] == source)].dropna()
    c = val[(val["source"] == source)].dropna()
    #gpt texts' length
    gpt_number = len(a[(a["label"]==1)]) + len(b[(b["label"]==1)]) + len(c[(c["label"]==1)])

    #human texts' length
    human_number = len(a[(a["label"]==0)]) + len(b[(b["label"]==0)]) + len(c[(c["label"]==0)])
    answers.append([human_number, gpt_number])
  return pd.DataFrame(answers, columns = [["human", "gpt"]])
# checking the category distribution and print it
m = check_category_numbers(test_data, val_data, train_data)
m

,human,gpt
0,51336,16660
1,1187,3561
2,842,842
3,1248,1337
4,3933,4503


# Preprocessing the data

In [ ]:
# install necessary libraries
import torch
from transformers import RobertaTokenizer
import nltk
from nltk.lm import Vocabulary
nltk.download('punkt_tab')# using nltk word tokenizer

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# remove the overapped texts whithin the datasets
test_data = test_data.drop_duplicates(subset = ["text"])
val_data = val_data.drop_duplicates(subset = ["text"])
train_data = train_data.drop_duplicates(subset = ["text"])

In [ ]:
# drop only 0,1,2,3 tokens over all datasets (removing None, no data, error messages)

# checking the number of tokens of the text and , if it is less than 4, remove the rows
def remove_less_tokens(df):
  def check_length(sentence):
    tokenized_sentence = nltk.tokenize.word_tokenize(str(sentence))
    return len(tokenized_sentence)

  df['length'] = df["text"].apply(check_length)
  df = df[df['length'] > 3]
  df = df[["text", "label", "source"]]
  return df

# applying the function above into train, validation, test data.
test_data = remove_less_tokens(test_data)
val_data = remove_less_tokens(val_data)
train_data = remove_less_tokens(train_data)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/tmp/ipykernel_14020/502022375.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['length'] = df["text"].apply(check_length)


In [ ]:
# checking the change of the category distribution after preprocessing.
m = check_category_numbers(test_data, val_data, train_data)
m

,human,gpt
0,47893,16302
1,1175,3517
2,797,842
3,1244,1306
4,3932,4465


In [ ]:
#store the data into csv file
val_data.to_csv('val.csv', index = False)
train_data.to_csv('train.csv', index = False)
test_data.to_csv('test.csv', index = False)

In [ ]:
# store the full data into csv file
full_data = pd.concat([test_data, val_data, train_data])
full_data.to_csv('full_preprocessed_data.csv', index = False)